In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import re
from scipy.stats import zscore

# Q1

In [ ]:
def read_dataset(path: str) -> pd.DataFrame:

    data = pd.read_csv(path)

    return data

df_superheros = read_dataset('data/superheros.csv')

In [ ]:
def ensure_all_different_name(df_superheros: pd.DataFrame) -> bool:

    return len(df_superheros) == df_superheros['name'].nunique()

ensure_all_different_name(df_superheros)

In [ ]:
def ensure_all_registers_unique(df_superheros: pd.DataFrame) -> bool:
    #Si la longitud del dataframe coincide la total con la longitud del dataframe contando solo los elementos únicos
    if len(df_superheros["name"]) == len(df_superheros["name"].nunique()): 
        return True 
    else: 
        return False 
    
ensure_all_registers_unique(df_superheros)

In [ ]:
def describe_data(df_superheros: pd.DataFrame):

    print(df_superheros.info())

    return

# Q2

In [ ]:
def find_doctor(name: str) -> bool:
    doctor_pattern = re.compile(r"((\w)+ [Dd]octor)|([Dd]r\. (\w)+)")
    match = doctor_pattern.search(name)

    is_doctor = False
    if match:
        is_doctor = True
    
    return is_doctor


def find_doctors(df_superheros: pd.DataFrame) -> pd.DataFrame:

    df_superheros['is_doctor'] = df_superheros['name'].apply(find_doctor)
    return df_superheros

In [ ]:
df_superheros_with_doctor = find_doctors(df_superheros)

In [ ]:
df_superheros_with_doctor['is_doctor'].value_counts()

In [ ]:
df_superheros_with_doctor.loc[df_superheros_with_doctor['is_doctor']]

# Q3

In [ ]:
def plot_boxplot(df_superheros: pd.DataFrame) -> None:

    df_superheros.boxplot('superpower')
    plt.show()

    return

plot_boxplot(df_superheros)

In [ ]:
def find_anomalous_superheros(df_superheros: pd.DataFrame)-> tuple[pd.DataFrame, list[str]]:
    mean_power = df_superheros.superpower.mean()
    std_power = df_superheros.superpower.std()
    df_superheros['z_score'] = (df_superheros['superpower'] - mean_power)/std_power

    anomalous_superheros = df_superheros.loc[abs(df_superheros["z_score"]) > 3].reset_index(drop=True).copy()

    df_superheros_clean = df_superheros.loc[abs(df_superheros["z_score"]) < 3].copy()

    df_superheros_clean.drop(columns='z_score', inplace=True)

    return (df_superheros_clean, list(anomalous_superheros['name'].values))


df_superheros_clean, anomalous_superheros = find_anomalous_superheros(df_superheros)
anomalous_superheros

In [ ]:
df_superheros_clean = pd.read_csv('data/superheros_clean.csv')

In [ ]:
df_superheros_clean

# Q4

In [ ]:
df_countries = pd.read_csv('data/countries.csv')

In [ ]:
def get_superheros_workplace(df_superheros_clean: pd.DataFrame, 
                             df_countries: pd.DataFrame) -> pd.DataFrame:

    df_superheros_workplace = df_superheros_clean.merge(df_countries, on='name')

    return df_superheros_workplace                    

In [ ]:
df_superheros_workplace = get_superheros_workplace(df_superheros_clean, df_countries)

In [ ]:
def aggregate_per_country(df_superheros_workplace: pd.DataFrame) -> pd.DataFrame:

    df_superheros_per_country = df_superheros_workplace.groupby('country').agg({'superpower': 'mean', 'name': 'count'}).reset_index()

    df_superheros_per_country.rename(columns={'name': 'n_superheros'}, inplace=True)
    
    return df_superheros_per_country

In [ ]:
df_superheros_per_country = aggregate_per_country(df_superheros_workplace)

In [ ]:
df_superheros_per_country

# Q5

In [ ]:
df_population_countries = pd.read_csv('data/countries_population.csv')

In [ ]:
def obtain_superheros_needed(df_superheros_workplace: pd.DataFrame, df_population_countries: pd.DataFrame):

    superheroes_spain = df_superheros_workplace.loc[(df_superheros_workplace['country'] == 'Spain') & 
                                                    (df_superheros_workplace['type'] == 'superhero'), 'name'].count()
    population_spain = df_population_countries.loc[df_population_countries['country'] == 'Spain', 'population'].sum()

    superheroes_per_capita_spain = superheroes_spain / population_spain

    population_china = df_population_countries.loc[df_population_countries['country'] == 'China', 'population'].sum()

    superheroes_needed_in_China = superheroes_per_capita_spain * population_china

    return int(superheroes_needed_in_China)

In [ ]:
obtain_superheros_needed(df_superheros_workplace, df_population_countries)